# Cross-Encoder + Cross-View + Penalty + Contrastive Loss — Netsol dataset

**Encoder của notebook này:** `BAAI/bge-reranker-v2-m3` — chạy **1 encoder × 4 fusion**.

Mỗi mẫu (1 file JSON) tách thành **50 cặp view** = 10 view CV × 5 view JD. Mỗi cặp đưa qua
cross-encoder (frozen) lấy CLS:

```
(cv_view_i [SEP] jd_view_j)  ->  Cross-Encoder(frozen)  ->  CLS  (50, hidden)  + mask(50)
   -> ViewPairScorer -> a_ij (50)  --mask-->  Fusion  ->  match score
```

- **mask**: view rỗng (`missing_section`) → 0, để loại khỏi fusion và bị penalty ép về 0.
- **4 fusion**: `weighted_sum`, `mlp`, `residual_mlp`, `attention`.
- **Loss** = BCE + λ·penalty(view rỗng) + α·activity(view có data) + λ_c·contrastive.

### Thích ứng cho Netsol
- Dataset tự chứa: `dataset_structuring/{train,val,test}/{matched,mismatch}/*.json`.
- Mỗi file có `resume_structured`, `job_description_structured`, `target_label` (0–10).
- **Nhãn nhị phân từ thư mục**: matched=1, mismatch=0 (dùng cho loss + Acc/P/R/F1).
- `yreg = target_label/10` (chỉ dùng báo cáo MAE/MSE).

In [ ]:
!pip install -q transformers sentencepiece einops

import os, json, gc, glob, zipfile
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             accuracy_score, precision_score, recall_score, f1_score)
from transformers.utils import logging as hf_logging; hf_logging.set_verbosity_error()
import warnings; warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| PyTorch:", torch.__version__)

Device: cuda | PyTorch: 2.10.0+cu128


In [ ]:
if os.path.exists("/kaggle"):
    HERE = "/kaggle/working"
    KAGGLE_ROOTS = ["/kaggle/input/datasets/hunthi/netsol/dataset_structuring"]
    KAGGLE_ROOTS += glob.glob("/kaggle/input/**/dataset_structuring", recursive=True)
    ZIP_CANDS = glob.glob("/kaggle/input/**/netsol_dataset.zip", recursive=True)
else:
    HERE = os.getcwd(); KAGGLE_ROOTS = []
    ZIP_CANDS = [os.path.join(HERE, "netsol_dataset.zip")]

WORK      = os.path.join(HERE, "outputs")
CACHE_DIR = os.path.join(WORK, "cache")
CKPT_DIR  = os.path.join(WORK, "checkpoints")
PLOT_DIR  = os.path.join(WORK, "plots")
RESULTS_CSV = os.path.join(WORK, "results_ce_crossview_pc.csv")
for d in (CACHE_DIR, CKPT_DIR, PLOT_DIR): os.makedirs(d, exist_ok=True)

def _find_root(base):
    if base is None: return None
    for cand in (base, os.path.join(base, "dataset_structuring")):
        if os.path.isdir(os.path.join(cand, "train")): return cand
    return None

DATA_ROOT = next((r for r in KAGGLE_ROOTS if _find_root(r)), None)
DATA_ROOT = _find_root(DATA_ROOT) or _find_root(HERE) or _find_root(WORK)
if DATA_ROOT is None:
    zp = next((z for z in ZIP_CANDS if os.path.exists(z)), None)
    assert zp, "Không tìm thấy dataset_structuring hoặc netsol_dataset.zip"
    with zipfile.ZipFile(zp) as z: z.extractall(WORK)
    DATA_ROOT = _find_root(WORK)
assert DATA_ROOT and os.path.isdir(os.path.join(DATA_ROOT, "train")), f"DATA_ROOT sai: {DATA_ROOT}"

# 10 view CV × 5 view JD -> 50 cặp
CV_VIEWS = ['skills','education','year_experience','experience','responsibilities',
            'projects','publications','achievement','certification','employment_history']
JD_VIEWS = ['skills','education','year_experience_requirement',
            'experience_requirement','responsibilities']
NUM_PAIRS = len(CV_VIEWS) * len(JD_VIEWS)   # 50

# >>> ENCODER CỦA NOTEBOOK NÀY (mỗi file 1 encoder) <<<
ENCODER_NAME = "bert-base-multilingual-cased"
# BAAI/bge-reranker-v2-m3
# jinaai/jina-reranker-v2-base-multilingual
# xlm-roberta-base
# bert-base-multilingual-cased

FUSIONS = ["weighted_sum", "mlp", "residual_mlp", "attention"]

MAX_LEN = 256
EPOCHS  = 150
BATCH   = 32
LR      = 1e-3
LABEL_SCALE = 10.0

LAMBDA_PENALTY     = 2.0
ALPHA_ACTIVITY     = 0.5
MARGIN_CONTRASTIVE = 0.5

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"Encoder  : {ENCODER_NAME} | fusions={FUSIONS}")
print(f"NUM_PAIRS={NUM_PAIRS} MAX_LEN={MAX_LEN} EPOCHS={EPOCHS} BATCH={BATCH}")

DATA_ROOT: /kaggle/input/datasets/hunthi/netsol/dataset_structuring
Encoder  : bert-base-multilingual-cased | fusions=['weighted_sum', 'mlp', 'residual_mlp', 'attention']
NUM_PAIRS=50 MAX_LEN=256 EPOCHS=150 BATCH=32


In [ ]:
def process_view(v):
    """missing_section -> ('[EMPTY_SECTION]', 0). Ngược lại -> (text, 1)."""
    if v is None: return "[EMPTY_SECTION]", 0
    if isinstance(v, list):
        c = [str(x) for x in v if str(x).lower() != "missing_section"]
        return (". ".join(c), 1) if c else ("[EMPTY_SECTION]", 0)
    s = str(v).lower()
    return ("[EMPTY_SECTION]", 0) if s in ("missing_section", "", "none") else (str(v), 1)

def build_rows(split):
    """matched/ (bin=1) + mismatch/ (bin=0). Mỗi file -> dict(cv_obj, jd_obj, yreg, ybin)."""
    rows = []
    for label, b in (("matched", 1), ("mismatch", 0)):
        for fp in sorted(glob.glob(os.path.join(DATA_ROOT, split, label, "*.json"))):
            with open(fp, "r", encoding="utf-8-sig") as f: d = json.load(f)
            rows.append({"cv_obj": d.get("resume_structured") or {},
                         "jd_obj": d.get("job_description_structured") or {},
                         "yreg": float(d.get("target_label", 0.0)) / LABEL_SCALE,
                         "ybin": float(b)})
    return rows

def label_to_str(b): return "matched" if b == 1 else "mismatch"

for sp in ("train", "val", "test"):
    r = build_rows(sp); npos = int(sum(x["ybin"] for x in r))
    print(f"  {sp:5s}: {len(r):3d} mẫu ({npos} matched / {len(r)-npos} mismatch)")
_ex = build_rows("train")[0]
print("CV views có data:", [v for v in CV_VIEWS if process_view(_ex['cv_obj'].get(v))[1]])
print("JD views có data:", [v for v in JD_VIEWS if process_view(_ex['jd_obj'].get(v))[1]])
print("yreg=", round(_ex['yreg'],3), "| ybin=", _ex['ybin'])

In [ ]:
def load_encoder(name):
    """Trả (tokenizer, encoder_frozen, hidden, slug). Xử lý jina custom-code + fallback XLM-R."""
    slug = name.split("/")[-1]
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if "jina" in name.lower():
        try:
            enc = AutoModel.from_pretrained(name, trust_remote_code=True, torch_dtype=torch.float32)
        except Exception as e:
            print(f"[{slug}] AutoModel custom lỗi -> fallback XLMRoberta: {e}")
            from transformers import XLMRobertaModel
            enc, info = XLMRobertaModel.from_pretrained(name, output_loading_info=True, torch_dtype=torch.float32)
            print(f"[{slug}] missing={len(info['missing_keys'])} unexpected={len(info['unexpected_keys'])}")
    else:
        enc = AutoModel.from_pretrained(name, trust_remote_code=True)
    enc = enc.to(device).eval()
    for p in enc.parameters(): p.requires_grad_(False)
    with torch.no_grad():
        _o = enc(**tok(["test"], ["test"], return_tensors="pt", truncation=True, max_length=8).to(device))
    assert hasattr(_o, "last_hidden_state"), f"{slug}: không có last_hidden_state"
    print(f"[{slug}] hidden={enc.config.hidden_size} | last_hidden_state OK")
    return tok, enc, enc.config.hidden_size, slug

In [ ]:
@torch.no_grad()
def embed_pairs_cls(cv_texts, jd_texts, tok, enc):
    """Tất cả cặp (cv_view_i, jd_view_j) -> CLS (num_pairs, hidden)."""
    a = [cv_texts[i] for i in range(len(cv_texts)) for _ in range(len(jd_texts))]
    b = [jd_texts[j] for _ in range(len(cv_texts)) for j in range(len(jd_texts))]
    cls = []
    for i in range(0, len(a), BATCH):
        t = tok(a[i:i+BATCH], b[i:i+BATCH], padding=True, truncation=True,
                max_length=MAX_LEN, return_tensors="pt").to(device)
        o = enc(**t)
        cls.append(o.last_hidden_state[:, 0, :].cpu().numpy().astype(np.float32))
    return np.concatenate(cls, 0)

def prepare_data(split, tok, enc, slug):
    """Mỗi mẫu -> CLS (50, hidden) + mask (50). Cache .npz theo slug+split (fail-fast nếu 0 mẫu)."""
    cache = os.path.join(CACHE_DIR, f"{slug}_{split}_ce.npz")
    if os.path.exists(cache):
        d = np.load(cache)
        print(f"  cache: {os.path.basename(cache)} | cls={d['cls'].shape}")
        return d["cls"], d["mask"], d["yreg"], d["ybin"]
    rows = build_rows(split)
    all_cls, all_mask, yreg, ybin = [], [], [], []
    n_skip = 0
    for r in tqdm(rows, desc=f"[{slug}] ce {split}"):
        try:
            cv_texts, cv_ex = zip(*[process_view(r["cv_obj"].get(v)) for v in CV_VIEWS])
            jd_texts, jd_ex = zip(*[process_view(r["jd_obj"].get(v)) for v in JD_VIEWS])
            cls = embed_pairs_cls(list(cv_texts), list(jd_texts), tok, enc)
            mask = np.outer(np.array(cv_ex, np.float32), np.array(jd_ex, np.float32)).flatten()
        except Exception as e:
            n_skip += 1; print(f"  !! bỏ qua 1 mẫu lỗi: {e}"); continue
        all_cls.append(cls); all_mask.append(mask)
        yreg.append(r["yreg"]); ybin.append(r["ybin"])
    if n_skip: print(f"  [{slug}|{split}] bỏ qua {n_skip}/{len(rows)} mẫu lỗi")
    if not all_cls:
        raise RuntimeError(f"[{slug}|{split}] không encode được mẫu nào -> không lưu cache rỗng")
    cls_arr = np.asarray(all_cls, np.float32); mask_arr = np.asarray(all_mask, np.float32)
    yreg = np.asarray(yreg, np.float32); ybin = np.asarray(ybin, np.float32)
    np.savez_compressed(cache, cls=cls_arr, mask=mask_arr, yreg=yreg, ybin=ybin)
    print(f"  saved: {os.path.basename(cache)} | cls={cls_arr.shape} mask={mask_arr.shape}")
    return cls_arr, mask_arr, yreg, ybin

In [ ]:
class CEDataset(Dataset):
    def __init__(self, cls, mask, yreg, ybin):
        self.cls  = torch.tensor(cls, dtype=torch.float32)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.yreg = torch.tensor(yreg, dtype=torch.float32).view(-1, 1)
        self.ybin = torch.tensor(ybin, dtype=torch.float32).view(-1, 1)
    def __len__(self): return len(self.cls)
    def __getitem__(self, i): return self.cls[i], self.mask[i], self.yreg[i], self.ybin[i]

class ViewPairScorer(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 1))
        nn.init.constant_(self.fc[3].bias, -2.0)
    def forward(self, cls): return torch.sigmoid(self.fc(cls)).squeeze(-1)

class WeightedSumFusion(nn.Module):
    def __init__(self, n=50): super().__init__(); self.layer = nn.Linear(n * 2, 1)
    def forward(self, a, mask): return torch.sigmoid(self.layer(torch.cat([a, mask], dim=-1)))

class FusionMLP(nn.Module):
    def __init__(self, n=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n * 2, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a, mask): return self.net(torch.cat([a, mask], dim=-1))

class ResidualBlock(nn.Module):
    def __init__(self, dim): super().__init__(); self.ln = nn.LayerNorm(dim); self.fc = nn.Linear(dim, dim)
    def forward(self, x): return x + F.relu(self.fc(self.ln(x)))

class FusionResidualMLP(nn.Module):
    def __init__(self, n=50):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(n * 2, 256), nn.ReLU())
        self.res1, self.res2 = ResidualBlock(256), ResidualBlock(256)
        self.hidden = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
                                    nn.Linear(128, 64), nn.ReLU())
        self.out = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, a, mask):
        x = self.inp(torch.cat([a, mask], dim=-1))
        x = self.res1(x); x = self.res2(x); x = self.hidden(x)
        return self.out(x)

class FusionAttention(nn.Module):
    def __init__(self, num_views=50, d_model=64, nhead=8):
        super().__init__()
        self.proj = nn.Linear(2, d_model)   # 1 -> 2: nhan ca score va mask
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=256, dropout=0.2, batch_first=True)
        self.attn = nn.TransformerEncoder(layer, num_layers=3)
        self.head = nn.Sequential(nn.Linear(d_model * num_views, 128), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(128, 1), nn.Sigmoid())
    def forward(self, a_ij, mask):
        b = a_ij.shape[0]
        x = torch.stack([a_ij, mask], dim=-1)   # (B,50,2) - them mask vao input
        h = self.proj(x)
        h = self.attn(h)
        return self.head(h.reshape(b, -1))

class FusionWithScorer(nn.Module):
    def __init__(self, hidden, fusion):
        super().__init__(); self.scorer = ViewPairScorer(hidden); self.fusion = fusion
    def forward(self, cls, mask):
        raw = self.scorer(cls); masked = raw * mask
        return self.fusion(masked, mask), masked, raw

FUSION_BUILDERS = {
    "weighted_sum": lambda: WeightedSumFusion(NUM_PAIRS),
    "mlp":          lambda: FusionMLP(NUM_PAIRS),
    "residual_mlp": lambda: FusionResidualMLP(NUM_PAIRS),
    "attention":    lambda: FusionAttention(NUM_PAIRS),
}

In [ ]:
LAMBDA_PENALTY     = 2.0
ALPHA_ACTIVITY     = 0.5
MARGIN_CONTRASTIVE = 0.5


def contrastive_loss(pred, ybin, margin=MARGIN_CONTRASTIVE):
    pos = ybin       * (1.0 - pred) ** 2
    neg = (1 - ybin) * F.relu(pred - margin) ** 2
    return (pos + neg).mean()


class TaskWeighter(nn.Module):
    """
    Hoc trong so dong cho 3 task loss: regression, classification, contrastive.
    Dense layer -> 3 neuron -> softmax -> (w_reg, w_cls, w_contra), sum=1.
    """
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1, 3)
        nn.init.constant_(self.fc.weight, 0.0)
        with torch.no_grad():
            self.fc.bias.copy_(torch.tensor([1.5, 0.5, 0.0]))  

    def forward(self):
        dummy = torch.ones(1, 1, device=next(self.parameters()).device)
        return torch.softmax(self.fc(dummy).squeeze(0), dim=0)   # (w_reg, w_cls, w_contra)


def _total_loss(pred, masked, raw, mask, yr, yb, crit_reg, crit_bin, w):
    l_reg    = crit_reg(pred, yr)
    l_cls    = crit_bin(pred, yb)
    l_contra = contrastive_loss(pred, yb)
    task     = w[0] * l_reg + w[1] * l_cls + w[2] * l_contra

    penalty  = torch.mean(((1 - mask) * raw) ** 2)
    activity = torch.mean(mask * (0.1 - masked).clamp(min=0))
    # penalty va activity la regularization co dinh, khong nam trong trong so hoc duoc
    return task + LAMBDA_PENALTY * penalty + ALPHA_ACTIVITY * activity


def train_one(model, train_loader, val_loader, ckpt, lr=1e-3):
    model    = model.to(device)
    task_w   = TaskWeighter().to(device)
    crit_reg = nn.MSELoss(); crit_bin = nn.BCELoss()
    opt   = optim.AdamW(
        list(model.parameters()) + list(task_w.parameters()),
        lr=lr, weight_decay=0.01
    )
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    hist  = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best  = float("inf")

    for ep in range(EPOCHS):
        model.train(); task_w.train(); tl = 0.; tp, tt = [], []
        for cls, mask, yr, yb in train_loader:
            cls, mask, yr, yb = cls.to(device), mask.to(device), yr.to(device), yb.to(device)
            opt.zero_grad()
            pred, masked, raw = model(cls, mask)
            w    = task_w()
            loss = _total_loss(pred, masked, raw, mask, yr, yb, crit_reg, crit_bin, w)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(model.parameters()) + list(task_w.parameters()), 1.0)
            opt.step(); tl += loss.item()
            tp += list((pred.detach().cpu().numpy().flatten() >= 0.5).astype(int))
            tt += list(yb.cpu().numpy().flatten().astype(int))

        model.eval(); task_w.eval(); vl = 0.; vp, vt = [], []
        with torch.no_grad():
            w = task_w()
            for cls, mask, yr, yb in val_loader:
                cls, mask, yr, yb = cls.to(device), mask.to(device), yr.to(device), yb.to(device)
                pred, masked, raw = model(cls, mask)
                vl += _total_loss(pred, masked, raw, mask, yr, yb, crit_reg, crit_bin, w).item()
                vp += list((pred.cpu().numpy().flatten() >= 0.5).astype(int))
                vt += list(yb.cpu().numpy().flatten().astype(int))

        av = vl / len(val_loader); sched.step(av)
        hist["train_loss"].append(tl / len(train_loader)); hist["val_loss"].append(av)
        hist["train_acc"].append(accuracy_score(tt, tp)); hist["val_acc"].append(accuracy_score(vt, vp))

        if av < best:
            best = av
            torch.save({"model": model.state_dict(), "task_w": task_w.state_dict()}, ckpt)

        if (ep + 1) % 10 == 0 or ep == 0:
            w_np = task_w().detach().cpu().numpy()
            print(f"    ep{ep+1:3d}/{EPOCHS} train_loss={hist['train_loss'][-1]:.4f} "
                  f"val_loss={av:.4f} val_acc={hist['val_acc'][-1]:.4f} | "
                  f"w_reg={w_np[0]:.3f} w_cls={w_np[1]:.3f} w_contra={w_np[2]:.3f}")

    return hist


def compute_metrics(y_reg, y_bin, preds):
    preds = np.asarray(preds); y_reg = np.asarray(y_reg); y_bin = np.asarray(y_bin)
    pb = (preds >= 0.5).astype(int)
    return {"MAE": mean_absolute_error(y_reg, preds), "MSE": mean_squared_error(y_reg, preds),
            "Accuracy": accuracy_score(y_bin, pb), "Precision": precision_score(y_bin, pb, zero_division=0),
            "Recall": recall_score(y_bin, pb, zero_division=0), "F1-Score": f1_score(y_bin, pb, zero_division=0)}


def evaluate(model, test_loader, ckpt):
    model = model.to(device)
    ckpt_data  = torch.load(ckpt, map_location=device)
    state = ckpt_data["model"] if "model" in ckpt_data else ckpt_data
    model.load_state_dict(state); model.eval()
    preds, yr, yb = [], [], []
    with torch.no_grad():
        for cls, mask, r, b in test_loader:
            pred, _, _ = model(cls.to(device), mask.to(device))
            preds += list(pred.cpu().numpy().flatten())
            yr += list(r.numpy().flatten()); yb += list(b.numpy().flatten())
    return compute_metrics(yr, yb, preds)


RESULT_COLS = ["Encoding Model", "Fusion", "MAE", "MSE", "Accuracy", "Precision", "Recall", "F1-Score",
               "w_reg", "w_cls", "w_contra"]

def append_result_csv(path, encoder, fusion, m):
    row = {"Encoding Model": encoder, "Fusion": fusion,
           "MAE": round(m["MAE"],4), "MSE": round(m["MSE"],4), "Accuracy": round(m["Accuracy"],4),
           "Precision": round(m["Precision"],4), "Recall": round(m["Recall"],4),
           "F1-Score": round(m["F1-Score"],4),
           "w_reg":    round(m.get("w_reg",    0), 4),
           "w_cls":    round(m.get("w_cls",    0), 4),
           "w_contra": round(m.get("w_contra", 0), 4)}
    df = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=RESULT_COLS)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df = df.drop_duplicates(subset=["Encoding Model", "Fusion"], keep="last")[RESULT_COLS]
    df.to_csv(path, index=False, encoding="utf-8-sig"); return df


def plot_history(hist, title, save_path):
    x = np.arange(1, len(hist["train_loss"]) + 1)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.plot(x, hist["train_loss"], label=f"Train (final={hist['train_loss'][-1]:.4f})")
    a1.plot(x, hist["val_loss"], label=f"Val (best={min(hist['val_loss']):.4f})")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title(f"{title} - Loss"); a1.legend(); a1.grid(True)
    a2.plot(x, hist["train_acc"], label=f"Train (final={hist['train_acc'][-1]:.4f})")
    a2.plot(x, hist["val_acc"], label=f"Val (best={max(hist['val_acc']):.4f})")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Accuracy"); a2.set_title(f"{title} - Acc"); a2.legend(); a2.grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show(); plt.close()

In [ ]:
print("="*80); print("ENCODER:", ENCODER_NAME); print("="*80)
tok, enc, hidden, slug = load_encoder(ENCODER_NAME)
cls_tr, m_tr, y_tr, b_tr = prepare_data("train", tok, enc, slug)
cls_v,  m_v,  y_v,  b_v  = prepare_data("val",   tok, enc, slug)
cls_te, m_te, y_te, b_te = prepare_data("test",  tok, enc, slug)
del enc, tok; gc.collect(); torch.cuda.empty_cache()
train_loader = DataLoader(CEDataset(cls_tr, m_tr, y_tr, b_tr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(CEDataset(cls_v,  m_v,  y_v,  b_v),  batch_size=BATCH)
test_loader  = DataLoader(CEDataset(cls_te, m_te, y_te, b_te), batch_size=BATCH)
for FUSION_NAME in FUSIONS:
    print(f"\n--- FUSION: {FUSION_NAME} ---")
    ckpt  = os.path.join(CKPT_DIR, f"{FUSION_NAME}_{slug}.pth")
    model = FusionWithScorer(hidden, FUSION_BUILDERS[FUSION_NAME]())
    hist  = train_one(model, train_loader, val_loader, ckpt, lr=LR_OVERRIDE.get(FUSION_NAME, LR))
    plot_history(hist, f"{FUSION_NAME} | {slug}", os.path.join(PLOT_DIR, f"{FUSION_NAME}-{slug}.png"))
    # Lay trong so cuoi cung tu checkpoint
    ckpt_data = torch.load(ckpt, map_location=device)
    if "task_w" in ckpt_data:
        task_w_final = TaskWeighter().to(device)
        task_w_final.load_state_dict(ckpt_data["task_w"])
        task_w_final.eval()
        with torch.no_grad():
            w_final = task_w_final().cpu().numpy()
        lambda_str = f"w_reg={w_final[0]:.4f} w_cls={w_final[1]:.4f} w_contra={w_final[2]:.4f}"
    else:
        w_final    = None
        lambda_str = "task_w not found"
    m = evaluate(model, test_loader, ckpt)
    if w_final is not None:
        m["w_reg"]    = round(float(w_final[0]), 4)
        m["w_cls"]    = round(float(w_final[1]), 4)
        m["w_contra"] = round(float(w_final[2]), 4)
    append_result_csv(RESULTS_CSV, ENCODER_NAME, FUSION_NAME, m)
    print(f"  TEST [{slug} | {FUSION_NAME}] MAE={m['MAE']:.4f} MSE={m['MSE']:.4f} "
          f"Acc={m['Accuracy']:.4f} P={m['Precision']:.4f} R={m['Recall']:.4f} "
          f"F1={m['F1-Score']:.4f} | {lambda_str}")
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
import zipfile

zip_ckpt = os.path.join(WORK, f"{slug}_checkpoints.zip")

with zipfile.ZipFile(zip_ckpt, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(CKPT_DIR):
        if fname.endswith(".pth"):
            zf.write(
                os.path.join(CKPT_DIR, fname),
                arcname=fname
            )

print(f"Checkpoints zipped: {zip_ckpt}")